# Optimal control intro: LQR vs value iteration for a pendulum swing-up

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/teaching/courses/gro860/labs/lqr_vs_vi_pendulum.ipynb)

This page shows a quick demo of two approaches for generating an "optimal" controller for a pendulum, based on the same quadratic cost function:

1. **LQR**: an analytical solution based on linearized dynamics, leading to a *local* linear control law.
2. **Value iteration (VI)**: a numerical solution based on dynamic programming over a discretized state space, leading to a *global* non-linear control law.

This page uses the toolbox [minilink](https://github.com/alx87grd/minilink).

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.control.lqr import lqr_at_operating_point
from minilink.core.costs import QuadraticCost
from minilink.core.diagram import DiagramSystem
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.pendulum.pendulum import Pendulum
from minilink.planning.policy_synthesis import plotting
from minilink.planning.policy_synthesis.discretizer import StateSpaceGrid
from minilink.planning.policy_synthesis.dp import (
    DynamicProgrammingOptions,
    DynamicProgrammingPlanner,
)
from minilink.planning.problems import PlanningProblem

## Defining a dynamic system model

Here we load an already defined pendulum class including all the dynamic equations. The target state is $\bar x = [\theta = -\pi, \dot\theta = 0]$, the upright position, which is also the linearization point used by the LQR.

In [ ]:
UPRIGHT = np.array([-np.pi, 0.0])  # target state and linearization point
TORQUE = 5.0  # actuator limits used by the VI solution


def make_pendulum():
    """Pendulum with bounded states and torque."""
    plant = Pendulum()
    plant.state.lower_bound = np.array([-2.0 * np.pi, -2.0 * np.pi])
    plant.state.upper_bound = np.array([+2.0 * np.pi, +2.0 * np.pi])
    plant.inputs["u"].lower_bound = np.array([-TORQUE])
    plant.inputs["u"].upper_bound = np.array([+TORQUE])
    return plant


plant = make_pendulum()

## Defining the cost function

Both controllers are synthesized using the same standard quadratic cost function of the type

$$J = \int \left( x' Q x + u' R u \right) dt$$

where the state error is taken with respect to the target $\bar x$.

In [ ]:
Q = np.diag([1.0, 1.0])
R = np.diag([1.0])

cost = QuadraticCost.from_system(plant, xbar=UPRIGHT, Q=Q, R=R)

print("Q=\n", cost.Q)
print("R=\n", cost.R)

## Synthesizing the "optimal" controllers

### LQR controller

Here we use a library function that:

1. linearizes the pendulum equations at the nominal state $\bar x$;
2. uses the obtained linearized equations and the defined cost function to compute the LQR controller solution $u = \bar u - K(x - \bar x)$.

In [ ]:
lqr_ctl = lqr_at_operating_point(make_pendulum(), UPRIGHT, Q, R)

K = lqr_ctl.params["K"]
print("LQR gain K =", np.round(K, 3))

### VI controller

Here we use library functions that:

1. discretize the domain of the states and control inputs of the system;
2. use value iteration to compute the optimal cost-to-go and control actions;
3. generate a continuous control law by interpolating in the computed discrete solution.

In [ ]:
problem = PlanningProblem(plant, x_goal=UPRIGHT, cost=cost)

grid = StateSpaceGrid(problem, x_grid_shape=(201, 201), u_grid_shape=(21,), dt=0.05)

planner = DynamicProgrammingPlanner(
    problem,
    grid=grid,
    options=DynamicProgrammingOptions(
        alpha=1.0, tol=0.1, max_iterations=2000, out_of_bound_cost=10000.0, verbose=True
    ),
)

result = planner.solve().policy
planner.clean_infeasible_set()

vi_ctl = result.controller()

## Showing the computed control laws

The next figures show maps illustrating the computed torque as a function of the two system states $\tau = \pi(\theta, \dot\theta)$ for both controllers.

In [ ]:
# LQR linear law evaluated on the same grid (clipped colormap at +/- TORQUE)
K_row = lqr_ctl.params["K"][0]
ubar = lqr_ctl.params["ubar"][0]
lqr_law = ubar - (grid.states - UPRIGHT) @ K_row

plotting.plot_value(
    grid, lqr_law, vmin=-TORQUE, vmax=TORQUE, cmap="bwr", title="LQR control law"
)

# VI policy
plotting.plot_policy(grid, result.pi)

We can see that the LQR solution (first figure) is a linear map, while the VI solution (second figure) is a non-linear map that follows the natural dynamics. Also note that the range of torques requested by the LQR solution is much larger. The LQR solution is optimal only locally, in the linear range around the target state, while the VI solution is the global optimal solution. If we zoom around the target state $[\theta = -\pi, \dot\theta = 0]$, locally both solutions tend to the same linear behaviour.

## Simulations

Here we show both control laws in action, with closed-loop trajectories starting near the bottom position $[\theta = -0.1, \dot\theta = 0]$.

In [ ]:
x0 = np.array([-0.1, 0.0])
tf = 10.0


def closed_loop(controller, x0, name):
    """Wire a state-feedback controller with a fresh pendulum plant."""
    plant = make_pendulum()
    plant.x0 = np.array(x0)
    diagram = DiagramSystem()
    diagram.add_subsystem(controller, "ctl")
    diagram.add_subsystem(plant, "plant")
    diagram.connect("plant", "y", "ctl", "x")
    diagram.connect("ctl", "u", "plant", "u")
    diagram.name = name
    diagram.camera_scale = 2.0
    traj = diagram.compute_trajectory(tf=tf, n_steps=2001)
    return diagram, plant, traj


cl_lqr, plant_lqr, traj_lqr = closed_loop(lqr_ctl, x0, "Pendulum with LQR")
cl_vi, plant_vi, traj_vi = closed_loop(vi_ctl, x0, "Pendulum with VI")

LQR:

In [ ]:
cl_lqr.plot_trajectory(traj_lqr)

VI:

In [ ]:
cl_vi.plot_trajectory(traj_vi)

We can see that both solutions converge to the target. The simulation with the LQR shows that the pendulum goes directly toward the goal, while the VI solution does a "pumping action" before swinging up toward the goal, in order to minimize the required torques. Note that the torque in the VI simulation is "noisy" because the VI algorithm outputs a discrete look-up table, which leads to this type of imperfection when converting back into a continuous domain.

## Animation of the simulations

LQR:

In [ ]:
cl_lqr.animate(traj_lqr)

VI (note the pumping action — this is one advantage of the VI algorithm: finding globally optimal solutions for non-linear systems):

In [ ]:
cl_vi.animate(traj_vi)

## Phase-plane trajectories

Here the same trajectories are shown on the phase plane of the pendulum. The vector field illustrates the natural dynamics along which the pendulum would evolve if no torque were applied. We can see why the VI solution requires less torque for the swing-up: it leverages the natural dynamics instead of fighting them with large torques.

In [ ]:
# LQR then VI
plant_lqr.plot_phase_plane(traj_lqr)
plant_vi.plot_phase_plane(traj_vi)

## Performance

Here the performance, in terms of the defined cost function $J = \int (x'Qx + u'Ru) \, dt$, is compared, as well as the maximum torque used by each solution.

In [ ]:
# Rebuild the applied torques from each control law
u_lqr = (ubar - (traj_lqr.x.T - UPRIGHT) @ K_row).reshape(1, -1)
u_vi = np.array([vi_ctl.action(x) for x in traj_vi.x.T]).T

traj_lqr_cost = cost.evaluate_trajectory(Trajectory(t=traj_lqr.t, x=traj_lqr.x, u=u_lqr))
traj_vi_cost = cost.evaluate_trajectory(Trajectory(t=traj_vi.t, x=traj_vi.x, u=u_vi))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(traj_lqr_cost.t, traj_lqr_cost.signals["cost"][0], label="LQR")
ax.plot(traj_vi_cost.t, traj_vi_cost.signals["cost"][0], label="VI")
ax.set_xlabel("t [s]")
ax.set_ylabel("$J = \\int (x'Qx + u'Ru) \\, dt$")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("LQR | total cost:", round(float(traj_lqr_cost.signals["cost"][0, -1]), 1),
      "| max torque:", round(float(np.abs(u_lqr).max()), 1), "Nm")
print("VI  | total cost:", round(float(traj_vi_cost.signals["cost"][0, -1]), 1),
      "| max torque:", round(float(np.abs(u_vi).max()), 1), "Nm")